# NB02: Co-occurrence Matrix and Statistical Testing

Computes Jaccard index, phi coefficient, Fisher's exact test, and permutation null
for pairwise associations between gene family groups.

Run equivalent script: `python src/02_cooccurrence_stats.py`

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import os
DATA_DIR = os.path.join('..', 'data')

In [2]:
df = pd.read_csv(os.path.join(DATA_DIR, 'species_gene_families.csv'))
print(f'Loaded {len(df)} species')

Loaded 27682 species


## Group-Level Co-occurrence

In [3]:
cooc = pd.read_csv(os.path.join(DATA_DIR, 'cooccurrence_matrix.csv'))
cooc[['group_A', 'group_B', 'n_both', 'jaccard', 'phi', 'odds_ratio', 'fisher_p']]

,group_A,group_B,n_both,jaccard,phi,odds_ratio,fisher_p
0,P_acquisition,N_fixation,4873,0.2102,0.0368,1.266,5.748275e-10
1,P_acquisition,Metal_handling,20683,0.7691,0.1096,2.304,1.284436e-65
2,P_acquisition,Phenazine,10308,0.4487,0.2616,5.196,0.000000e+00
3,P_acquisition,Phenazine_operon,63,0.0028,0.0238,inf,1.515331e-06
4,N_fixation,Metal_handling,5718,0.2239,0.1067,4.042,1.517761e-87
5,N_fixation,Phenazine,2252,0.1531,-0.0183,0.912,2.442921e-03
6,N_fixation,Phenazine_operon,3,0.0005,-0.0192,0.185,5.268461e-04
7,Metal_handling,Phenazine,10632,0.4113,0.1224,2.859,6.658950e-100
8,Metal_handling,Phenazine_operon,63,0.0025,0.0144,inf,9.368977e-03
9,Phenazine,Phenazine_operon,63,0.0057,0.0584,inf,8.633351e-26


## Permutation Test Results

1,000 permutations shuffling group membership vectors.

In [4]:
groups = {
    'P_acquisition': ['phoA', 'phoD_pfam', 'pstA', 'pstB', 'pstC', 'pstS', 'phnC', 'phnD', 'phnE'],
    'N_fixation': ['nifH', 'nifD', 'nifH_pfam'],
    'Metal_handling': ['copA', 'corA', 'feoB_pfam', 'HMA_pfam'],
    'Phenazine_operon': None,
}

group_has = {}
for gname, genes in groups.items():
    if genes is None:
        group_has[gname] = df['has_phz_operon'].values
    else:
        cols = [f'has_{g}' for g in genes if f'has_{g}' in df.columns]
        group_has[gname] = (df[cols].sum(axis=1) >= 1).astype(int).values

def phi_coefficient(a, b):
    n11 = np.sum((a==1)&(b==1)); n10 = np.sum((a==1)&(b==0))
    n01 = np.sum((a==0)&(b==1)); n00 = np.sum((a==0)&(b==0))
    denom = np.sqrt((n11+n10)*(n01+n00)*(n11+n01)*(n10+n00))
    return (n11*n00 - n10*n01) / denom if denom > 0 else 0.0

np.random.seed(42)
N_PERM = 1000
key_pairs = [('P_acquisition','Metal_handling'),('N_fixation','Metal_handling'),
             ('Phenazine_operon','Metal_handling'),('P_acquisition','N_fixation')]

for g1, g2 in key_pairs:
    a, b = group_has[g1], group_has[g2]
    obs_phi = phi_coefficient(a, b)
    null_phis = np.array([phi_coefficient(a, np.random.permutation(b)) for _ in range(N_PERM)])
    z = (obs_phi - np.mean(null_phis)) / np.std(null_phis) if np.std(null_phis) > 0 else float('inf')
    perm_p = (np.sum(np.abs(null_phis) >= np.abs(obs_phi)) + 1) / (N_PERM + 1)
    print(f'{g1} vs {g2}: phi={obs_phi:.4f}, Z={z:.1f}, perm_p={perm_p:.4f}')

P_acquisition vs Metal_handling: phi=0.1096, Z=17.7, perm_p=0.0010


N_fixation vs Metal_handling: phi=0.1067, Z=18.4, perm_p=0.0010


Phenazine_operon vs Metal_handling: phi=0.0144, Z=2.3, perm_p=0.0170


P_acquisition vs N_fixation: phi=0.0368, Z=6.1, perm_p=0.0010


## Per-Gene-Family Pairwise Detail

In [5]:
detail = pd.read_csv(os.path.join(DATA_DIR, 'pairwise_detail.csv'))
print('Top 10 enrichments (nutrient x metal):')
detail.sort_values('enrichment', ascending=False).head(10)[['nutrient_gene','metal_gene','enrichment','phi','n_both','fisher_p']]

Top 10 enrichments (nutrient x metal):


,nutrient_gene,metal_gene,enrichment,phi,n_both,fisher_p
47,nifH_pfam,HMA_pfam,1.957,0.2292,1988,3.300946e-284
46,nifH_pfam,feoB_pfam,1.895,0.2685,2641,0.000000e+00
65,phzG,corA,1.795,0.0569,116,5.252337e-27
61,phzD,corA,1.780,0.0280,29,3.295366e-07
53,phzA,corA,1.726,0.0190,15,1.508052e-03
7,phoD_pfam,HMA_pfam,1.720,0.0842,468,6.252962e-40
73,phzM,corA,1.710,0.0174,13,4.997763e-03
75,phzM,HMA_pfam,1.704,0.0092,6,1.315791e-01
57,phzB,corA,1.649,0.0295,43,2.352989e-07
6,phoD_pfam,feoB_pfam,1.527,0.0772,570,9.226759e-36


In [6]:
print('Bottom 5 (depleted co-occurrence):')
detail[detail['phi']<0].sort_values('phi').head(5)[['nutrient_gene','metal_gene','enrichment','phi','n_both','fisher_p']]

Bottom 5 (depleted co-occurrence):


,nutrient_gene,metal_gene,enrichment,phi,n_both,fisher_p
18,pstC,feoB_pfam,0.668,-0.2559,3386,0.000000e+00
22,pstS,feoB_pfam,0.641,-0.1838,2038,6.741474e-214
19,pstC,HMA_pfam,0.719,-0.1733,2655,2.655168e-183
50,phzF,feoB_pfam,0.721,-0.1625,2701,1.233050e-164
10,pstA,feoB_pfam,0.863,-0.1438,5573,2.540281e-124
